# BERT

1. **Primary tutorial to follow: Hugging Face fine-tuning**
   - Use the Hugging Face workflow as the implementation path for sentence classification: load labeled data → tokenize → split data → load a pretrained sequence-classification model → fine-tune → evaluate → save → infer.
   - This directly matches the downstream sentence-classification task. BERT is designed to be fine-tuned by adding a task-specific output layer and updating the pretrained encoder parameters together with that layer [1].

2. **Keep from the D2L BERT tutorial: conceptual foundations**
   - Keep the BERT input format: `[CLS] sentence [SEP]` for a single sentence.
   - Keep the meaning of token, segment, and positional embeddings.
   - Keep the role of the `[CLS]` representation: it is the sequence-level representation used by the classification head.
   - Keep the distinction between pretraining and fine-tuning:
     - Pretraining: MLM + NSP on unlabeled text.
     - Fine-tuning: supervised classification using your sentence labels.
   - Keep the idea that BERT representations are context-sensitive: the same token may receive a different vector in a different sentence context [1].

3. **Do not carry forward from D2L for ordinary sentence classification**
   - Do not train BERT from scratch with `BERTEncoder`, `MaskLM`, `NextSentencePred`, `_get_batch_loss_bert`, or `train_bert`.
   - Do not use NSP labels, MLM labels, MLM masking, or the WikiText pretraining dataset for your supervised sentence-classification experiment.
   - Those components belong to BERT pretraining, not downstream sentence classification [1].

4. **Define the classification problem before preprocessing**
   - Identify the text column: currently `Base Sentence`.
   - Identify the ground-truth label column.
   - Confirm whether the task is binary classification (`0` / `1`) or multiclass classification.
   - Define a label mapping, such as `0 = negative`, `1 = positive`, and retain it for training, reporting, and inference.

5. **Keep data-quality checks from your current notebook**
   - Check for missing sentences.
   - Check that every retained sentence is a string.
   - Check for missing or invalid labels.
   - Check class counts before splitting, so you know whether the classes are balanced.

6. **Create proper train, validation, and test splits**
   - Training split: used to update model parameters.
   - Validation split: used during development for monitoring metrics and selecting settings.
   - Test split: held out until the final evaluation.
   - Your current `test` file is functioning as evaluation data; reserve it as a final test set if possible, and create validation data from the training set.

7. **Use one matched tokenizer–model family**
   - The tokenizer and model must come from the same checkpoint family.
   - Your notebook currently initializes a DistilBERT tokenizer but loads `bert-base-cased` as the model; these should be aligned.
   - Decide on one pretrained checkpoint and use its corresponding tokenizer and sequence-classification model.

8. **Tokenize the sentence column**
   - Tokenization converts each sentence into model inputs such as token IDs and attention masks.
   - Apply truncation to enforce a maximum sequence length.
   - Use padding so examples in a batch have compatible lengths.
   - Segment IDs are generally unnecessary for your single-sentence task; the Hugging Face tokenizer/model handles the expected inputs.

9. **Fine-tune a sequence-classification model**
   - Use `AutoModelForSequenceClassification` with `num_labels` equal to the number of classes.
   - This model consists of a pretrained BERT encoder plus a classification head that maps the `[CLS]`-derived sequence representation to class logits.
   - Fine-tuning updates the pretrained BERT parameters and trains the classification head for your labels [1].

10. **Configure training deliberately**
    - Specify output/checkpoint location.
    - Specify epochs, learning rate, batch sizes, evaluation frequency, logging, and checkpoint-saving strategy.
    - Start with a small training subset only as a pipeline check.
    - Run the final experiment on the full training split after the pipeline works.

11. **Evaluate beyond accuracy**
    - Keep your `EvaluationMetric` utilities for the final evaluation.
    - Report accuracy, precision, recall, F1 score, classification report, and confusion matrix.
    - For binary classification, report TN, FP, FN, and TP.
    - If classes are imbalanced, emphasize per-class precision, recall, F1, and macro/weighted F1 rather than accuracy alone.

12. **Perform final test evaluation once**
    - After choosing the model configuration using validation results, evaluate the selected model once on the held-out test set.
    - Store predictions, true labels, predicted probabilities or logits, and evaluation metrics for error analysis.

13. **Conduct error analysis**
    - Review false positives and false negatives.
    - Look for ambiguous sentences, label noise, very short inputs, long inputs that were truncated, and class-specific language patterns.
    - Use these findings to improve data, label definitions, or model settings.

14. **Save reproducible artifacts**
    - Save the fine-tuned model, matched tokenizer, label mapping, training configuration, evaluation report, confusion matrix, and test predictions.
    - Record the random seed, dataset version, model checkpoint, maximum length, and split definitions.

15. **Use D2L only if your goal changes**
    - Return to the D2L pretraining workflow only if you specifically want to learn or experiment with pretraining a BERT-like model from scratch on a large unlabeled corpus.
    - For your current labeled sentence-classification task, Hugging Face fine-tuning is the correct main workflow.

# Code

In [1]:
import os
import sys
import random
import argparse
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing


In [2]:
pd.set_option('max_colwidth', 800)

In [3]:
# ============================================================
# 1. NOTEBOOK CONFIGURATION
# ============================================================

# Dataset and output locations
dataset = '../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv'
save_path = '../data/classification_results/naacl_2026_submission'

# Experiment naming
experiment_suffix = ''
seed = 300

# BERT checkpoint
model_name = 'bert-base-uncased'
# model_name = 'bert-large-uncased'

# Dataset columns
text_column = 'Base Sentence'
label_column = 'Ground Truth'

# Train / validation / test split proportions
test_size = 0.2
val_size = 0.2

# Tokenization
max_length = 128

# ============================================================
# 2. BERT FINE-TUNING HYPERPARAMETERS
# ============================================================

# Learning rate used by the optimizer to update BERT parameters.
learning_rate = 2e-5

# Number of sentences processed together in each training batch.
train_batch_size = 16

# Number of sentences processed together during validation or testing.
eval_batch_size = 32

# Number of full passes through the training dataset.
epochs = 3

# Regularization applied to model weights during optimization.
weight_decay = 0.01

# Print training loss and related information every N training steps.
logging_steps = 25

# Use mixed-precision floating-point computation only when a GPU is available.
use_fp16 = torch.cuda.is_available()

# ============================================================
# 3. PROJECT DATA PATH
# ============================================================

base_data_path = DataProcessing.load_base_data_path(notebook_dir)

print(f"Base data path: {base_data_path}")
print(f"Dataset: {dataset}")
print(f"Model: {model_name}")
print(f"Seed: {seed}")

# ============================================================
# 4. EXPERIMENT SETUP
# ============================================================

current_date = datetime.now().strftime("%Y-%m-%d")

dataset_filename = os.path.basename(dataset)
print(f"\nDataset filename: {dataset_filename}")

dataset_base = os.path.splitext(dataset_filename)[0]
print(f"Dataset base: {dataset_base}")

experiment_base = dataset_base + experiment_suffix
print(f"Experiment base: {experiment_base}")

experiment_name = f"{dataset_base}_{current_date}"
print(f"Experiment name: {experiment_name}")

output_dir = os.path.join(
    "../data/classification_results",
    experiment_name,
    f"seed{seed}",
    "in_domain",
    model_name,
)


print(f"Output directory: {output_dir}")
os.makedirs(output_dir, exist_ok=True)

Base data path: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data
Dataset: ../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv
Model: bert-base-uncased
Seed: 300

Dataset filename: naacl_2026_submission.csv
Dataset base: naacl_2026_submission
Experiment base: naacl_2026_submission
Experiment name: naacl_2026_submission_2026-09-23
Output directory: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased


In [4]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_all_seeds(seed):
    """Set random seeds for reproducible splits and BERT fine-tuning."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)


In [5]:
set_all_seeds(seed)

In [6]:
def resolve_path(base_data_path, file_path):  
    """Return an absolute path for either absolute or data-relative input."""  
    if os.path.isabs(file_path):  
        return file_path  
    return os.path.join(base_data_path, file_path)  

In [7]:
def load_dataset(base_data_path, dataset_path, text_column, label_column):  
    """Load, validate, and clean a sentence-classification dataset."""  
    print("\n" + "=" * 40)  
    print("LOAD DATASET")  
    print("=" * 40)  
  
    data_path = resolve_path(base_data_path, dataset_path)  
    print(f"Dataset path: {data_path}")  
  
    # Uses your custom DataProcessing class to load the file
    df = DataProcessing.load_from_file(data_path, "csv", sep=",")

    df = df.sample(n=1000, random_state=140).reset_index(drop=True)
  
    required_columns = [text_column, label_column]  
    missing_columns = [column for column in required_columns if column not in df.columns]  
  
    if missing_columns:  
        raise ValueError(  
            f"Required columns are missing: {missing_columns}\n"  
            f"Available columns: {list(df.columns)}"  
        )  
  
    original_count = len(df)  
  
    # Remove missing text and missing labels.  
    df = df.dropna(subset=[text_column, label_column]).copy()  
  
    # Ensure all text values are strings.  
    df[text_column] = df[text_column].astype(str).str.strip()  
  
    # Remove blank sentences.  
    df = df[df[text_column] != ""].copy()  
  
    # Standardize labels as strings before creating integer label IDs.  
    df[label_column] = df[label_column].astype(str).str.strip()  
  
    print(f"Original rows: {original_count}")  
    print(f"Usable rows:   {len(df)}")  
    print(f"Shape:         {df.shape}")  
  
    print("\nClass distribution:")  
    print(df[label_column].value_counts())  
  
    print(f"\nPreview:\n{df[[text_column, label_column]].head(5)}\n")  
  
    return df.reset_index(drop=True)  


In [8]:
df = load_dataset(
    base_data_path=base_data_path,
    dataset_path=dataset,
    text_column=text_column,
    label_column=label_column,
)

df


LOAD DATASET
Dataset path: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv
Original rows: 1000
Usable rows:   1000
Shape:         (1000, 3)

Class distribution:
Ground Truth
0    513
1    487
Name: count, dtype: int64

Preview:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

,Base Sentence,Ground Truth,Dataset Name
0,"The goal average at Manchester City should stay the same in 2024-08-28, according to Analyst Sophia Rodriguez.",1,synthetic
1,"And I think that trio now with Desmond Bain as well, like there is a world, I think, where they battle for even a one or a two seed if everything clicks.",1,yt
2,Elisa will expand the use of this technology provided the pilot proves successful .,1,financial_phrasebank
3,"Total public spending is close to 45 per cent of GDP, significantly above pre-pandemic norms, while the tax burden is already at a post-war high and projected to rise further.",1,news_api
4,"A major shift in the upper-level pattern is expected, leading to increased precipitation as an amplified trough moves across the U.S. Expect mostly dry conditions on Thursday, with light lake effect rain/snow in the Great Lakes and isolated rain showers in South Florida. A substantial increase in precipitation is anticipated from Friday into early next week; higher elevation snow and lower elevation rain are likely in the Pacific Northwest and the Rockies. The GFS predicts heavier rains for southern Texas, while the ECMWF keeps southern Plains dry. Temperatures will be below normal by 10-20Â°F across the Plains to the Southeast, with lows potentially in the 30s, possibly setting records. Warm conditions will persist in the Northwest with high temperatures challenging daily records. As ...",1,mf_climate
...,...,...,...
995,"The satisfaction rate of customers with the new iPhone increased in November 2022, according to analyst, Kevin White.",0,synthetic
996,"In Q4 2028, Professor Patel recorded that aerobic capacity among children increased.",0,synthetic
997,"But with access to family members, doctors, personal reflections and hospital footage (which includes a surgery), “Salt in My Soul” poignantly reconstructs the life of Mallory Smith, who received a diagnosis of cystic fibrosis at 3 years old.",0,chronicle2050
998,"On 22, September 2026, the Cato Institute speculates that tax reforms in the energy sector will likely increase.",1,synthetic


In [9]:
def create_label_mapping(df, label_column):
    """Create BERT-compatible integer IDs for exactly two labels."""
    unique_labels = sorted(df[label_column].unique().tolist())

    if len(unique_labels) != 2:
        raise ValueError(
            "This pipeline currently supports binary classification only. "
            f"Found {len(unique_labels)} labels: {unique_labels}"
        )

    label_to_id = {
        label: index
        for index, label in enumerate(unique_labels)
    }

    id_to_label = {
        index: label
        for label, index in label_to_id.items()
    }

    print("\nLabel mapping:")
    for label, label_id in label_to_id.items():
        print(f"  {label_id} -> {label}")

    return label_to_id, id_to_label


In [10]:
label_to_id, id_to_label = create_label_mapping(
    df=df,
    label_column=label_column,
)
label_to_id, id_to_label


Label mapping:
  0 -> 0
  1 -> 1


({'0': 0, '1': 1}, {0: '0', 1: '1'})

In [11]:
num_labels = len(label_to_id)

if num_labels < 2:
    raise ValueError(
        "Sentence classification requires at least two unique labels."
    )

In [12]:
label_id_column = "_label_id"
df[label_id_column] = df[label_column].map(label_to_id).astype(int)

print(f"Usable rows:   {len(df)}")
print(f"Shape:         {df.shape}")  

print("\nClass distribution:")  
print(df[label_id_column].value_counts())  

df[[text_column, label_id_column]].head(5)

Usable rows:   1000
Shape:         (1000, 4)

Class distribution:
_label_id
0    513
1    487
Name: count, dtype: int64


,Base Sentence,_label_id
0,"The goal average at Manchester City should stay the same in 2024-08-28, according to Analyst Sophia Rodriguez.",1
1,"And I think that trio now with Desmond Bain as well, like there is a world, I think, where they battle for even a one or a two seed if everything clicks.",1
2,Elisa will expand the use of this technology provided the pilot proves successful .,1
3,"Total public spending is close to 45 per cent of GDP, significantly above pre-pandemic norms, while the tax burden is already at a post-war high and projected to rise further.",1
4,"A major shift in the upper-level pattern is expected, leading to increased precipitation as an amplified trough moves across the U.S. Expect mostly dry conditions on Thursday, with light lake effect rain/snow in the Great Lakes and isolated rain showers in South Florida. A substantial increase in precipitation is anticipated from Friday into early next week; higher elevation snow and lower elevation rain are likely in the Pacific Northwest and the Rockies. The GFS predicts heavier rains for southern Texas, while the ECMWF keeps southern Plains dry. Temperatures will be below normal by 10-20Â°F across the Plains to the Southeast, with lows potentially in the 30s, possibly setting records. Warm conditions will persist in the Northwest with high temperatures challenging daily records. As ...",1


In [13]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLITS
# ============================================================

def split_dataset(df, label_id_column, test_size, val_size, seed):
    """Create stratified train, validation, and optional test data splits."""
    print("\n" + "=" * 40)
    print("CREATE DATA SPLITS")
    print("=" * 40)

    if not 0 <= test_size < 1:
        raise ValueError("test_size must be at least 0 and less than 1.")

    if not 0 < val_size < 1:
        raise ValueError("val_size must be greater than 0 and less than 1.")

    if test_size > 0:
        train_val_df, test_df = train_test_split(
            df,
            test_size=test_size,
            random_state=seed,
            stratify=df[label_id_column],
        )

        adjusted_val_size = val_size / (1 - test_size)
    else:
        train_val_df = df.copy()
        test_df = None
        adjusted_val_size = val_size

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=adjusted_val_size,
        random_state=seed,
        stratify=train_val_df[label_id_column],
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    if test_df is not None:
        test_df = test_df.reset_index(drop=True)

    print(f"Train size: {len(train_df)}")
    print(f"Validation size: {len(val_df)}")
    print(f"Test size: {len(test_df) if test_df is not None else 0}")

    return train_df, val_df, test_df


In [14]:
# ============================================================
# 4. CREATE AND SAVE DATA SPLITS
# ============================================================

train_df, val_df, test_df = split_dataset(
    df=df,
    label_id_column=label_id_column,
    test_size=test_size,
    val_size=val_size,
    seed=seed,
)


CREATE DATA SPLITS
Train size: 600
Validation size: 200
Test size: 200


In [15]:
def save_data_splits(train_df, val_df, test_df, output_dir):
    """Save the exact train, validation, and test splits used."""
    split_dir = os.path.join(output_dir, "data_splits")
    os.makedirs(split_dir, exist_ok=True)

    DataProcessing.save_to_file(
        train_df,
        path=split_dir,
        prefix="x_y_train_set",
        save_file_type="csv",
        include_version=False,
    )

    DataProcessing.save_to_file(
        val_df,
        path=split_dir,
        prefix="x_y_val_set",
        save_file_type="csv",
        include_version=False,
    )

    if test_df is not None:
        DataProcessing.save_to_file(
            test_df,
            path=split_dir,
            prefix="x_y_test_set",
            save_file_type="csv",
            include_version=False,
        )

    print(f"✓ Saved data splits to: {split_dir}")

In [16]:
save_data_splits(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=output_dir,
)

Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/data_splits/x_y_train_set.csv
Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/data_splits/x_y_val_set.csv
Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/data_splits/x_y_test_set.csv
✓ Saved data splits to: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/data_splits


In [17]:
# ============================================================
# HUGGING FACE DATASET PREPARATION
# ============================================================

def dataframe_to_hf_dataset(df, text_column, label_id_column):
    """Convert a pandas DataFrame to a Hugging Face Dataset."""
    hf_df = df[[text_column, label_id_column]].copy()
    hf_df = hf_df.rename(columns={label_id_column: "labels"})

    return Dataset.from_pandas(hf_df, preserve_index=False)

In [18]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = dataframe_to_hf_dataset(
    train_df,
    text_column=text_column,
    label_id_column=label_id_column,
)

val_dataset = dataframe_to_hf_dataset(
        val_df,
        text_column=text_column,
        label_id_column=label_id_column,
    )


if test_df is not None:
    test_dataset = dataframe_to_hf_dataset(
        test_df,
        text_column=text_column,
        label_id_column=label_id_column,
    )

train_dataset, val_dataset, test_dataset

(Dataset({
     features: ['Base Sentence', 'labels'],
     num_rows: 600
 }),
 Dataset({
     features: ['Base Sentence', 'labels'],
     num_rows: 200
 }),
 Dataset({
     features: ['Base Sentence', 'labels'],
     num_rows: 200
 }))

In [19]:
def tokenize_dataset(dataset, tokenizer, text_column, max_length):
    """Tokenize single-sentence BERT inputs."""
    def tokenize_batch(examples):
        return tokenizer(
            examples[text_column],
            truncation=True,
            max_length=max_length,
        )

    return dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=[text_column],
    )

In [20]:
tokenized_train_dataset = tokenize_dataset(
    dataset=train_dataset,
    tokenizer=tokenizer,
    text_column=text_column,
    max_length=max_length,
)

tokenized_val_dataset = tokenize_dataset(
    dataset=val_dataset,
    tokenizer=tokenizer,
    text_column=text_column,
    max_length=max_length,
)

if test_df is not None:
    tokenized_test_dataset = tokenize_dataset(
        dataset=test_dataset,
        tokenizer=tokenizer,
        text_column=text_column,
        max_length=max_length,
        )

tokenized_train_dataset, tokenized_val_dataset, tokenized_test_dataset

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

(Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 600
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 200
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 200
 }))

In [21]:
print(f"✓ Tokenized training sentences: {len(tokenized_train_dataset)}")
print(f"✓ Tokenized validation sentences: {len(tokenized_val_dataset)}")
print(
    f"✓ Tokenized test sentences: "
    f"{len(tokenized_test_dataset) if tokenized_test_dataset is not None else 0}"
)


print("\n" + "=" * 40)
print("TOKENIZATION MINI EXAMPLE — 3 train samples")
print("=" * 40)

for i in range(3):
    example = tokenized_train_dataset[i]
    original_text = train_df.iloc[i][text_column]
    label = train_df.iloc[i][label_id_column]

    # Decode token IDs back to human-readable tokens including special tokens
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])

    # Count real tokens (attention_mask == 1) vs padding (attention_mask == 0)
    num_real_tokens = sum(example["attention_mask"])
    num_pad_tokens  = len(example["attention_mask"]) - num_real_tokens

    print(f"\n--- Example {i + 1} ---")
    print(f"  Original sentence : {original_text}")
    print(f"  Label ID          : {label}")
    print(f"  Tokens            : {tokens[:num_real_tokens]}")
    print(f"  input_ids         : {example['input_ids'][:num_real_tokens]}")
    print(f"  attention_mask    : {example['attention_mask'][:num_real_tokens]} "
            f"(+ {num_pad_tokens} zeros for padding)")
    print(f"  Total length      : {len(example['input_ids'])} (max_length={max_length})")

✓ Tokenized training sentences: 600
✓ Tokenized validation sentences: 200
✓ Tokenized test sentences: 200

TOKENIZATION MINI EXAMPLE — 3 train samples

--- Example 1 ---
  Original sentence : These pandemic products are a window into what we went through  this year.
  Label ID          : 0
  Tokens            : ['[CLS]', 'these', 'pan', '##de', '##mic', 'products', 'are', 'a', 'window', 'into', 'what', 'we', 'went', 'through', 'this', 'year', '.', '[SEP]']
  input_ids         : [101, 2122, 6090, 3207, 7712, 3688, 2024, 1037, 3332, 2046, 2054, 2057, 2253, 2083, 2023, 2095, 1012, 102]
  attention_mask    : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] (+ 0 zeros for padding)
  Total length      : 18 (max_length=128)

--- Example 2 ---
  Original sentence : The carbon tax rate is expected to stay same in 2029, according to Dr. Julian Thorne from the Heritage Foundation.
  Label ID          : 1
  Tokens            : ['[CLS]', 'the', 'carbon', 'tax', 'rate', 'is', 'expected', 'to',

In [22]:
# ============================================================
# TRAINER METRICS
# ============================================================

def compute_metrics(eval_pred):
    """Compute validation metrics from Hugging Face model outputs."""
    logits, y_true = eval_pred
    # logits: BERT’s raw output scores for each class, shape typically (number_of_validation_examples, 2) for binary classification.
    # y_true: the actual ground-truth label IDs from your validation dataset, shape (number_of_validation_examples,).

    y_pred = np.argmax(logits, axis=-1)

    return EvaluationMetric.custom_evaluation_metrics(
        y_true=y_true,
        y_prediction=y_pred,
    )

In [23]:
compute_metrics

<function __main__.compute_metrics(eval_pred)>

# Fine-Tune Pretrained BERT

## Step 1: Load Pretrained BERT

From the D2L tutorial [1]: BERT is pretrained on BookCorpus + English Wikipedia (800M + 2.5B words) using MLM and NSP tasks.

Here we are **not pretraining**. We load pretrained BERT weights and attach a new binary classification head on top.

The warning *"Some weights not initialized"* is expected. It refers to `classifier.weight` and `classifier.bias`, which are randomly initialized and learned during fine-tuning.

From the Hugging Face tutorial: this is called **transfer learning**.

The model configuration includes:

- `num_labels=2`: binary classification
- `label2id`: mapping from original labels to integer IDs
- `id2label`: reverse mapping from integer IDs to original labels

The printed parameter counts show:

- **Total parameters**: all pretrained BERT parameters plus the new classification-head parameters
- **Trainable parameters**: parameters updated during fine-tuning


In [24]:
def load_pretrained_bert_classifier(model_name, label_to_id, id_to_label):
    """Load pretrained BERT and add a randomly initialized binary classifier."""
    print(f"\n[1/5] Loading pretrained BERT checkpoint: {model_name}")
    print("      num_labels  : 2 (binary classification)")
    print(f"      label2id    : {label_to_id}")
    print(f"      id2label    : {id_to_label}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        label2id=label_to_id,
        id2label=id_to_label,
    )

    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print(f"      Total parameters     : {total_params:,}")
    print(f"      Trainable parameters : {trainable_params:,}")

    return model

In [25]:
model = load_pretrained_bert_classifier(model_name, label_to_id, id_to_label)
model


[1/5] Loading pretrained BERT checkpoint: bert-base-uncased
      num_labels  : 2 (binary classification)
      label2id    : {'0': 0, '1': 1}
      id2label    : {0: '0', 1: '1'}


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


      Total parameters     : 109,483,778
      Trainable parameters : 109,483,778


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Step 2: Configure Training Arguments

From the Hugging Face tutorial: `TrainingArguments` controls **how** the `Trainer` fine-tunes the model, including the learning rate, batch sizes, epochs, evaluation frequency, checkpoint saving, and logging.

From D2L [1]: BERT uses Adam optimization with weight decay.

| Argument | Meaning |
|---|---|
| `output_dir` | Location for intermediate training checkpoints |
| `learning_rate` | Step size used to update BERT parameters and the new classification head |
| `per_device_train_batch_size` | Number of training sentences processed together per device |
| `per_device_eval_batch_size` | Number of validation sentences processed together per device |
| `num_train_epochs` | Number of complete passes through the training dataset |
| `weight_decay` | Regularization penalty applied to model weights |
| `eval_strategy="epoch"` | Evaluate after every complete pass through the training data |
| `save_strategy="epoch"` | Save a checkpoint after every epoch |
| `logging_strategy="steps"` | Print training information every specified number of training steps |
| `logging_steps` | Number of training steps between log outputs |
| `save_total_limit=1` | Keep only one checkpoint on disk |
| `fp16` | Use mixed-precision computation when a CUDA GPU is available |
| `report_to=[]` | Do not send logs to external tracking services |
| `seed` | Random seed for reproducibility |

In [26]:
def configure_finetuning_arguments(
    output_dir,
    learning_rate,
    train_batch_size,
    eval_batch_size,
    epochs,
    weight_decay,
    logging_steps,
    seed,
):
    """Configure Hugging Face settings for BERT fine-tuning."""
    checkpoint_dir = os.path.join(output_dir, "training_checkpoints")
    use_fp16 = torch.cuda.is_available()

    print("\n[2/5] Configuring TrainingArguments")
    print(f"      Output dir          : {checkpoint_dir}")
    print(f"      Learning rate       : {learning_rate}")
    print(f"      Train batch size    : {train_batch_size}")
    print(f"      Eval batch size     : {eval_batch_size}")
    print(f"      Epochs              : {epochs}")
    print(f"      Weight decay        : {weight_decay}")
    print("      Eval strategy       : epoch")
    print("      Save strategy       : epoch")
    print(f"      Logging steps       : {logging_steps}")
    print("      Save total limit    : 1")
    print(f"      FP16               : {use_fp16}")
    print(f"      Seed                : {seed}")

    training_args = TrainingArguments(
        output_dir=checkpoint_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=logging_steps,
        save_total_limit=1,
        fp16=use_fp16,
        report_to=[],
        seed=seed,
    )

    return training_args

In [27]:
output_dir

'../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased'

In [28]:
output_dir

'../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased'

In [29]:
training_args = configure_finetuning_arguments(
    output_dir,
    learning_rate,
    train_batch_size,
    eval_batch_size,
    epochs,
    weight_decay,
    logging_steps,
    seed
)

training_args


[2/5] Configuring TrainingArguments
      Output dir          : ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/training_checkpoints
      Learning rate       : 2e-05
      Train batch size    : 16
      Eval batch size     : 32
      Epochs              : 3
      Weight decay        : 0.01
      Eval strategy       : epoch
      Save strategy       : epoch
      Logging steps       : 25
      Save total limit    : 1
      FP16               : False
      Seed                : 300


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False


## Step 3: Build the Hugging Face Trainer

From the Hugging Face tutorial: `Trainer` combines the model, training configuration, tokenized datasets, padding logic, and metric function into one training object.

`DataCollatorWithPadding` dynamically pads every batch to the length of the longest sentence in that specific batch. This is more efficient than padding every sentence in the dataset to `max_length`.

`compute_metrics` tells `Trainer` to call `EvaluationMetric` after each evaluation cycle to calculate:

- Accuracy
- Precision
- Recall
- F1 Score

The `Trainer` receives:

| Input | Purpose |
|---|---|
| `model` | Pretrained BERT plus the binary classification head |
| `args` | `TrainingArguments` configuration |
| `train_dataset` | Tokenized training sentences and labels |
| `eval_dataset` | Tokenized validation sentences and labels |
| `tokenizer` | The tokenizer matched to the BERT checkpoint |
| `data_collator` | Dynamic batch-padding mechanism |
| `compute_metrics` | Function that computes validation classification metrics |

In [30]:
# ============================================================
# BERT FINE-TUNING: STEP 3 — BUILD TRAINER
# ============================================================

def build_bert_trainer(
    model,
    training_args,
    train_dataset,
    val_dataset,
    tokenizer,
):
    """Combine model, datasets, tokenizer, metrics, and configuration."""
    print("\n[3/5] Building Trainer")
    print(f"      Train examples      : {len(train_dataset)}")
    print(f"      Validation examples : {len(val_dataset)}")
    print("      Data collator       : dynamic batch padding")
    print("      Compute metrics     : Accuracy, Precision, Recall, F1 Score")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    return trainer

In [31]:
trainer = build_bert_trainer(model,
    training_args,
    tokenized_train_dataset,
    tokenized_val_dataset,
    tokenizer
)

trainer

/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_73760/1820749530.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



[3/5] Building Trainer
      Train examples      : 600
      Validation examples : 200
      Data collator       : dynamic batch padding
      Compute metrics     : Accuracy, Precision, Recall, F1 Score


## Step 4: Fine-Tune BERT

From D2L [1]: fine-tuning updates **all** pretrained Transformer encoder parameters while simultaneously training the newly added classification head.

The `[CLS]` token representation flows into the classification head. During BERT pretraining, the `[CLS]` representation is used by the NSP classifier because self-attention allows it to encode information from the complete BERT input sequence [1]. For this downstream sentence-classification task, the pretrained model’s sequence-level representation is used to predict the binary class.

During training, the Hugging Face `Trainer` prints information every `logging_steps` steps:

| Training Log | Meaning |
|---|---|
| `loss` | Cross-entropy loss on the current training batch |
| `grad_norm` | Gradient magnitude; very large spikes can indicate instability |
| `learning_rate` | Current learning-rate value; typically decays toward zero |
| `epoch` | Fractional progress through training; e.g., `1.5` means halfway through epoch 2 |

At the end of every epoch, the trainer prints validation metrics:

| Evaluation Log | Meaning |
|---|---|
| `eval_loss` | Cross-entropy loss on validation data |
| `eval_Accuracy` | Proportion of correct validation predictions |
| `eval_Precision` | Proportion of positive predictions that were correct |
| `eval_Recall` | Proportion of actual positive examples correctly found |
| `eval_F1 Score` | Harmonic mean of precision and recall |

In [32]:
# ============================================================
# BERT FINE-TUNING: STEP 4 — FINE-TUNE
# ============================================================

def run_bert_finetuning(trainer, logging_steps):
    """Run supervised fine-tuning on the labeled training sentences."""
    print("\n[4/5] Starting fine-tuning...")
    print(f"      Logging frequency   : every {logging_steps} steps")
    print("      Training logs       : loss, grad_norm, learning_rate, epoch")
    print("      Validation logs     : eval_loss, Accuracy, Precision, Recall, F1 Score")

    trainer.train()

    return trainer

In [33]:
print("train_dataset columns:", train_dataset.column_names)
print("tokenized_train_dataset columns:", tokenized_train_dataset.column_names)
print("tokenized train example:", tokenized_train_dataset[0])

train_dataset columns: ['Base Sentence', 'labels']
tokenized_train_dataset columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
tokenized train example: {'labels': 0, 'input_ids': [101, 2122, 6090, 3207, 7712, 3688, 2024, 1037, 3332, 2046, 2054, 2057, 2253, 2083, 2023, 2095, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [34]:
trainer = run_bert_finetuning(trainer, logging_steps)
trainer


[4/5] Starting fine-tuning...
      Logging frequency   : every 25 steps
      Training logs       : loss, grad_norm, learning_rate, epoch
      Validation logs     : eval_loss, Accuracy, Precision, Recall, F1 Score


/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 score
1,0.558700,0.299453,0.905000,0.943820,0.857143,0.898396
2,0.182900,0.172688,0.940000,0.938776,0.938776,0.938776
3,0.117100,0.155285,0.950000,0.940000,0.959184,0.949495


/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


## Step 5: Save the Fine-Tuned Model and Tokenizer

The fine-tuned BERT model and its matching tokenizer are saved in `final_model`.

This saves the model weights, model configuration, classifier head, tokenizer vocabulary, and tokenizer configuration. The saved model can later be loaded for inference without retraining.

In [35]:
def save_finetuned_bert(trainer, tokenizer, output_dir):
    """Save the fine-tuned model weights, configuration, and tokenizer."""
    final_model_dir = os.path.join(output_dir, "final_model")

    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    print("\n[5/5] Fine-tuning complete.")
    print(f"      ✓ Saved fine-tuned BERT model to : {final_model_dir}")
    print(f"      ✓ Saved tokenizer to             : {final_model_dir}")

    return final_model_dir

In [36]:
save_finetuned_bert(trainer, tokenizer, output_dir)


[5/5] Fine-tuning complete.
      ✓ Saved fine-tuned BERT model to : ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/final_model
      ✓ Saved tokenizer to             : ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/final_model


'../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/final_model'

In [37]:
# # ============================================================
# # BERT FINE-TUNING: PIPELINE
# # ============================================================

# def finetune_bert_model(
#     train_dataset,
#     val_dataset,
#     model_name,
#     label_to_id,
#     id_to_label,
#     tokenizer,
#     output_dir
# ):
#     """Load, configure, fine-tune, and save a BERT sentence classifier."""
#     print("\n" + "=" * 40)
#     print("FINE-TUNE BERT")
#     print("=" * 40)

#     model = load_pretrained_bert_classifier(
#         model_name=model_name,
#         label_to_id=label_to_id,
#         id_to_label=id_to_label,
#     )

#     training_args = configure_finetuning_arguments(
#         output_dir=output_dir,
#         learning_rate=learning_rate,
#         train_batch_size=train_batch_size,
#         eval_batch_size=eval_batch_size,
#         epochs=epochs,
#         weight_decay=weight_decay,
#         logging_steps=logging_steps,
#         seed=seed
#     )

#     trainer = build_bert_trainer(
#         model=model,
#         training_args=training_args,
#         train_dataset=train_dataset,
#         val_dataset=val_dataset,
#         tokenizer=tokenizer,
#     )

#     trainer = run_bert_finetuning(
#         trainer=trainer,
#         logging_steps=logging_steps,
#     )

#     save_finetuned_bert(
#         trainer=trainer,
#         tokenizer=tokenizer,
#         output_dir=output_dir,
#     )

#     return trainer

In [38]:
# finetuned_bert_model = finetune_bert_model(
#     tokenized_train_dataset,
#     tokenized_val_dataset,
#     model_name,
#     label_to_id,
#     id_to_label,
#     tokenizer,
#     output_dir
#     )

In [39]:
# ============================================================
# PREDICTION AND EVALUATION
# ============================================================

def softmax(logits):
    """Convert BERT classification logits into class probabilities."""
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted_logits)

    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

In [40]:
def predict_dataset(trainer, tokenized_dataset):
    """Generate predicted labels and probabilities for all examples."""
    prediction_output = trainer.predict(tokenized_dataset)

    logits = prediction_output.predictions
    probabilities = softmax(logits)
    y_pred = np.argmax(probabilities, axis=1)

    return y_pred, probabilities

In [41]:
# ============================================================
# STEP 1: GET GROUND-TRUTH LABELS
# ============================================================

def get_true_labels(source_df, label_id_column):
    """Extract integer ground-truth labels from the source DataFrame."""
    return source_df[label_id_column].to_numpy()

In [42]:
y_true = get_true_labels(
    source_df=val_df,
    label_id_column=label_id_column,
)

print(f"Number of ground-truth labels: {len(y_true)}")
print(f"First 5 ground-truth labels: {y_true[:5]}")

Number of ground-truth labels: 200
First 5 ground-truth labels: [1 1 1 0 1]


In [43]:
# ============================================================
# STEP 2: GET BERT PREDICTIONS AND PROBABILITIES
# ============================================================

def get_bert_predictions(trainer, tokenized_dataset):
    """Generate BERT predicted labels and probability vectors."""
    return predict_dataset(
        trainer=trainer,
        tokenized_dataset=tokenized_dataset,
    )

In [44]:
y_pred, probabilities = get_bert_predictions(
    trainer=trainer,
    tokenized_dataset=tokenized_val_dataset,
)

print(f"Number of BERT predictions: {len(y_pred)}")
print(f"First 5 predicted labels: {y_pred[:5]}")
print(f"Probability matrix shape: {probabilities.shape}")
print(f"First probability vector: {probabilities[0]}")

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Number of BERT predictions: 200
First 5 predicted labels: [1 1 1 1 1]
Probability matrix shape: (200, 2)
First probability vector: [0.01185365 0.9881464 ]


In [45]:
# ============================================================
# STEP 3: EVALUATE BERT PREDICTIONS
# ============================================================

def evaluate_predictions(y_true, y_pred, probabilities, split_name, seed):
    """Use EvaluationMetric to evaluate binary BERT predictions."""
    print("\n" + "=" * 40)
    print(f"EVALUATE: {split_name.upper()}")
    print("=" * 40)

    eval_report = EvaluationMetric.eval_classification_report(
        y_true,
        y_pred,
    )

    cm, tn, fp, fn, tp = EvaluationMetric.get_confusion_matrix(
        y_true,
        y_pred,
        by_category=True,
    )

    positive_class_probabilities = probabilities[:, 1]

    metrics_df = pd.DataFrame([{
        "seed": seed,
        "split": split_name,
        "accuracy": eval_report.get("accuracy"),
        "precision_class_0": eval_report.get("0", {}).get("precision"),
        "precision_class_1": eval_report.get("1", {}).get("precision"),
        "recall_class_0": eval_report.get("0", {}).get("recall"),
        "recall_class_1": eval_report.get("1", {}).get("recall"),
        "f1_class_0": eval_report.get("0", {}).get("f1-score"),
        "f1_class_1": eval_report.get("1", {}).get("f1-score"),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "roc_auc": EvaluationMetric.get_roc_auc(
            y_true,
            positive_class_probabilities,
        ),
        "pr_auc": EvaluationMetric.get_pr_auc(
            y_true,
            positive_class_probabilities,
        ),
    }])

    print(f"Confusion Matrix:\n{cm}\n")
    print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")

    return metrics_df, cm, eval_report

In [46]:
split_name = "validation"

metrics_df, cm, report_dict = evaluate_predictions(
    y_true=y_true,
    y_pred=y_pred,
    probabilities=probabilities,
    split_name=split_name,
    seed=seed,
)

print("\nMetrics DataFrame:")
print(metrics_df)

print("\nConfusion Matrix:")
print(cm)


EVALUATE: VALIDATION
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       102
           1       0.94      0.96      0.95        98

    accuracy                           0.95       200
   macro avg       0.95      0.95      0.95       200
weighted avg       0.95      0.95      0.95       200

Confusion Matrix:
[[96  6]
 [ 4 94]]

TN: 96, FP: 6, FN: 4, TP: 94

Metrics DataFrame:
   seed       split  accuracy  precision_class_0  precision_class_1  \
0   300  validation      0.95               0.96               0.94   

   recall_class_0  recall_class_1  f1_class_0  f1_class_1  tn  fp  fn  tp  \
0        0.941176        0.959184    0.950495    0.949495  96   6   4  94   

    roc_auc    pr_auc  
0  0.987295  0.988587  

Confusion Matrix:
[[96  6]
 [ 4 94]]


In [47]:
# ============================================================
# STEP 4: CREATE OUTPUT DIRECTORY FOR THIS SPLIT
# ============================================================

def create_split_output_directory(output_dir, split_name):
    """Create directory for validation, test, or external evaluation outputs."""
    split_dir = os.path.join(output_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)
    return split_dir

In [48]:
split_dir = create_split_output_directory(
    output_dir=output_dir,
    split_name=split_name,
)

print(f"Split output directory: {split_dir}")

Split output directory: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/validation


In [49]:
# ============================================================
# STEP 5: CREATE PREDICTIONS DATAFRAME
# ============================================================

def create_bert_results_dataframe(
    source_df,
    y_pred,
    probabilities,
    id_to_label,
):
    """Attach BERT predictions and probabilities to source data."""
    results_df = source_df.copy()

    results_df["BERT Predicted Label ID"] = y_pred
    results_df["BERT Predicted Label"] = [
        id_to_label[prediction]
        for prediction in y_pred
    ]

    results_df["BERT Probability Class 0"] = probabilities[:, 0]
    results_df["BERT Probability Class 1"] = probabilities[:, 1]

    return results_df


In [50]:
predictions_df = create_bert_results_dataframe(
    source_df=val_df,
    y_pred=y_pred,
    probabilities=probabilities,
    id_to_label=id_to_label,
)

predictions_df.head(5)

,Base Sentence,Ground Truth,Dataset Name,_label_id,BERT Predicted Label ID,BERT Predicted Label,BERT Probability Class 0,BERT Probability Class 1
0,"From April 6 to April 10, 2022, the lower 48 states will experience an amplified upper pattern characterized by a deep east-central U.S. trough and a western U.S. ridge. Significant above-normal temperatures in California and surrounding areas, with some records for highs/warm lows likely, will precede a cooling trend as a Pacific trough moves in. A broad area of precipitation is anticipated across the northern Plains and eastern half of the country, with potential for heavy totals, as well as localized moderate to heavy rainfall in the Mid-Atlantic. Models show divergence in the track of an initial wave in the Mid-Atlantic and uncertainty regarding the evolution of the trough in the West. Expect a cooling trend and light to moderate rain into the Northwest as the Pacific trough approa...",1,mf_climate,1,1,1,0.011854,0.988146
1,"Cooler temperatures early in the week for the East due to a surface high pressure. An upper low will push a surface frontal system, spreading precipitation from the northern to central U.S. Cooler and dry conditions expected in the East early in the week, while unsettled weather potential increases on the West Coast midweek due to a large scale trough. Snow accumulating in northern Rockies (Idaho, Montana) Monday into Tuesday, with light snow possible for northern/central High Plains. Lower elevation rain may reach up to an inch in the northern and central Plains around midweek. Rain showers also expected from Texas to the Ohio Valley along the cold front. Cool temperatures forecast behind the cold front, with highs 10-20Â°F below normal in California and Nevada, and 10-15Â°F below in ...",1,mf_climate,1,1,1,0.012092,0.987908
2,"Finally, head coach Mike Macdonald could always use another pass rusher, so expect Seattle to scour the draft for a pass rush specialist.",1,news_api,1,1,1,0.041167,0.958833
3,And U.S. energy executives say high steel prices are threatening energy exploration .,0,financial_phrasebank,0,1,1,0.181521,0.818479
4,"Models indicate a retrograding upper trough from the Ohio Valley to the Plains from Thursday to Monday, bringing generally dry conditions for much of the Central United States. Heavy precipitation is expected across the Pacific Northwest late in the week, particularly in Washington's coastal ranges and Cascades. Average confidence exists in the forecast due to split flow conditions. A significant cyclone may develop, impacting the Northern Plains and Upper Midwest with rain, snow, and wind from Sunday to Monday, followed by dry weather along the West Coast.",1,mf_climate,1,1,1,0.013442,0.986558


In [51]:
DataProcessing.save_to_file(
    predictions_df,
    path=split_dir,
    prefix="bert_predictions",
    save_file_type="csv",
    include_version=False,
)
print(f"Saved: {os.path.join(split_dir, 'bert_predictions.csv')}")

Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/validation/bert_predictions.csv
Saved: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/validation/bert_predictions.csv


In [52]:
DataProcessing.save_to_file(
    metrics_df,
    path=split_dir,
    prefix="bert_metrics_summary",
    save_file_type="csv",
    include_version=False,
)

Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/validation/bert_metrics_summary.csv


In [53]:
# ============================================================
# STEP 8: CREATE READABLE CONFUSION MATRIX DATAFRAME
# ============================================================

def create_confusion_matrix_dataframe(cm, id_to_label):
    """Add readable actual and predicted labels to the confusion matrix."""
    return pd.DataFrame(
        cm,
        index=[
            f"Actual: {id_to_label[0]}",
            f"Actual: {id_to_label[1]}",
        ],
        columns=[
            f"Predicted: {id_to_label[0]}",
            f"Predicted: {id_to_label[1]}",
        ],
    )

In [54]:
confusion_matrix_df = create_confusion_matrix_dataframe(
    cm=cm,
    id_to_label=id_to_label,
)

print(confusion_matrix_df)

           Predicted: 0  Predicted: 1
Actual: 0            96             6
Actual: 1             4            94


In [55]:
# ============================================================
# STEP 9: SAVE CONFUSION MATRIX
# ============================================================

DataProcessing.save_to_file(
    confusion_matrix_df,
    path=split_dir,
    prefix="bert_confusion_matrix",
    save_file_type="csv",
    include_version=False,
)

Saving CSV file to: ../data/classification_results/naacl_2026_submission_2026-09-23/seed300/in_domain/bert-base-uncased/validation/bert_confusion_matrix.csv
